# Confirming the ELM eta saturation result on zephyr-7b-beta

`FINDINGS.md` §1 shows that the ELM erase target collapses to a one-hot
distribution well before the reference's default `--eta 1000`, measured on
Qwen2.5-0.5B-Instruct. This notebook repeats that measurement on the paper's
actual base model, **zephyr-7b-beta**, which needs a CUDA GPU.

**Runtime → Change runtime type → T4 GPU** before running. Free tier is enough:
the model is loaded in 4-bit (~4.5 GB) and this is inference only, no training.

Expected total time: ~10 minutes, most of it the model download.

The number to compare against, from 0.5B:

| eta | mean max prob | mean entropy | mean tokens with mass |
|---|---|---|---|
| 1 | 0.42 | 3.59 | 9,823 |
| 20 | 0.82 | 0.75 | 877 |
| 100 | 0.94 | 0.15 | 6.8 |
| 1000 | 0.99 | 0.02 | 1.1 |

If the 7B numbers land in the same regime, the finding holds for the released
model and the claim in FINDINGS.md §1 can drop its caveat.

In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets

import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# Self-contained copy of elm/targets.py::combine_logprobs, so this notebook
# needs no repo checkout. Keep in sync if the source changes.
import torch

def combine_logprobs(original, expert, novice, eta_start, eta_end, top_k=None):
    n_positions, vocab = original.shape
    eta = torch.linspace(eta_start, eta_end, n_positions,
                         device=original.device, dtype=original.dtype).unsqueeze(1)
    edit = original + eta * (novice - expert)
    if top_k is not None:
        k = min(top_k, vocab)
        kth = edit.topk(k, dim=-1).values[:, -1:]
        edit = edit.masked_fill(edit < kth, float('-inf'))
    return torch.softmax(edit, dim=-1)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'HuggingFaceH4/zephyr-7b-beta'

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant, device_map='auto')
model.eval()
print('loaded. vocab', model.config.vocab_size)

In [ ]:
# Same corpus and personas the reimplementation uses. Neither WMDP cyber split
# is gated, so this needs no HF token.
from datasets import load_dataset
import random

CONCEPT = 'exploit development, malware analysis, reverse engineering, penetration testing, vulnerability research'
EXPERT = f'Here is a text written by an expert in the field of {CONCEPT}, with detailed technical information and all the knowledge:\n'
NOVICE = f'The text is written by a novice, with no knowledge about {CONCEPT} and steering the conversation to random fun topics:\n'

forget = load_dataset('cais/wmdp-corpora', 'cyber-forget-corpus', split='train')['text']
forget = [t[:700] for t in forget if len(t) > 50]
passages = random.Random(0).sample(forget, 8)
print(len(forget), 'passages available')

In [ ]:
TEMPERATURE = 1.2
MAX_TOKENS = 192
ETAS = [0.5, 1, 2, 5, 10, 20, 50, 100, 1000]

def logprobs(text):
    ids = tokenizer(text, return_tensors='pt', truncation=True,
                    max_length=MAX_TOKENS + 64).to(model.device)
    with torch.no_grad():
        logits = model(**ids).logits[0].float()
    return torch.log_softmax(logits / TEMPERATURE, dim=-1)

rows = {e: [] for e in ETAS}
guidance_std = []

for text in passages:
    original, expert, novice = logprobs(text), logprobs(EXPERT + text), logprobs(NOVICE + text)
    n = min(original.shape[0], MAX_TOKENS)
    original, expert, novice = original[:n], expert[-n:], novice[-n:]
    guidance_std.append((novice - expert).std().item())
    for eta in ETAS:
        t = combine_logprobs(original, expert, novice, eta, eta, top_k=None)
        safe = t.clamp_min(1e-30)
        rows[eta].append((
            t.max(dim=-1).values.mean().item(),
            -(safe * safe.log()).sum(dim=-1).mean().item(),
            (t > 1e-6).sum(dim=-1).float().mean().item(),
        ))

print(f'{MODEL_ID} | vocab {model.config.vocab_size:,} | {len(passages)} passages')
print(f'std of (novice - expert) = {sum(guidance_std)/len(guidance_std):.3f}   (0.5B measured 2.145)\n')
print(f"{'eta':>6} {'mean max prob':>14} {'mean entropy':>13} {'mean support':>13}")
print('-' * 50)
for eta in ETAS:
    mp = sum(r[0] for r in rows[eta]) / len(rows[eta])
    en = sum(r[1] for r in rows[eta]) / len(rows[eta])
    su = sum(r[2] for r in rows[eta]) / len(rows[eta])
    print(f'{eta:>6} {mp:>14.6f} {en:>13.4f} {su:>13.1f}')

## What to look for

The claim in `FINDINGS.md` §1 holds on 7B if:

1. **mean support at eta=1000 is in the low single digits.** That is the core
   claim: the target is effectively a one-hot label, so `--use_erase_soft_loss`
   and `--topk 50` have nothing to act on.
2. **mean support at eta=100 is well under 50.** That is what makes `--topk 50`
   inert, and with the 1→1000 ramp eta passes 100 by roughly token 19 of 192.
3. **entropy is non-monotonic**, rising from the eta=0 baseline at small eta
   before collapsing. That is what makes an ablation in the eta 1–5 range
   interesting rather than obvious.

Paste the printed table into `FINDINGS.md` §1 next to the 0.5B numbers.